# Inferential Statistics: Thống kê suy diễn
**STAT3013 | TrainHyp**

| Mục | Chi tiết |
|-----|---------|
| Input  | `data_features.csv` (từ NB01) |
| Output | Kết quả kiểm định thống kê (T-test, ANOVA, Chi-square) |

## Lý do khoa học cho Thống kê suy diễn
> **Thống kê suy diễn là công cụ để chuyển từ "quan sát" sang "chứng minh khoa học".**

Sự khác biệt cốt lõi:
- **Thống kê mô tả:** Chỉ cho thấy nhóm tập cường độ cao có vẻ tăng cơ tốt hơn.
- **Thống kê suy diễn:** Dùng các phép toán (p-value, T-test, ANOVA) để chứng minh sự khác biệt đó là có ý nghĩa thống kê thực sự, chứ không phải do ăn may hay ngẫu nhiên.

Tại sao bước này quan trọng:
- Giúp bác bỏ giả thuyết không (H0) một cách khách quan.
- Xác định chắc chắn xem các yếu tố như khối lượng tập luyện (Volume), kinh nghiệm (Trained/Untrained) hay việc tập đến ngưỡng thất bại (Failure) có thực sự tác động đến kết quả phì đại cơ bắp hay không.

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from scipy import stats

In [3]:
df = pd.read_csv("/content/gdrive/MyDrive/STAT3013/Dataset/data_cleaned.csv")

# [MỤC ĐÍCH]: Ép kiểu Ordinal cho Khối lượng tập. Giúp mô hình và các hàm kiểm định hiểu được logic tăng tiến tuyến tính của sinh lý học (Dose-response: Low < Moderate < High).
df["volume_category"] = pd.Categorical(df["volume_category"],
    categories=["Low","Moderate","High"], ordered=True)

# [RÀO CHẮN THỐNG KÊ]: Chốt cứng ngưỡng sai lầm loại I (Alpha = 5%) làm thước đo tiêu chuẩn để đối chiếu p-value và ra quyết định cho toàn bộ các kiểm định phía sau.
ALPHA = 0.05

print(f"Loaded: {df.shape[0]} rows")
print(f"Alpha = {ALPHA}")

Loaded: 198 rows
Alpha = 0.05


## 1. t-test: Low vs High Volume (hedges_g)
**H0:** Mean hedges_g của Low = High

**H1:** Khác nhau có ý nghĩa thống kê

In [4]:
# [MỤC ĐÍCH]: So sánh hiệu ứng phì đại giữa hai thái cực Khối lượng tập (Low vs High).
# [ĐIỂM CHẠM HỌC THUẬT]: Kiểm chứng quy luật Dose-Response. Theo giả thuyết, High Volume thường dẫn đến kết quả (mean) cao hơn nhưng cũng đi kèm độ biến thiên (SD) lớn hơn do rủi ro quá tải.
low  = df[df["volume_category"] == "Low"]["hedges_g"].dropna()
high = df[df["volume_category"] == "High"]["hedges_g"].dropna()

# Xuất thống kê mô tả làm căn cứ so sánh độ dốc (Magnitude) của hiệu ứng trước khi chạy các kiểm định chuyên sâu.
print(f"Low:  n={len(low)},  mean={low.mean():.4f}, SD={low.std():.4f}")
print(f"High: n={len(high)}, mean={high.mean():.4f}, SD={high.std():.4f}")

Low:  n=48,  mean=0.3051, SD=0.3871
High: n=89, mean=0.5416, SD=0.3803


In [5]:
# [RÀO CHẮN THỐNG KÊ]: Kiểm định tính đồng nhất phương sai giữa hai mức Volume (Low vs High).
# Dùng center="median" để đảm bảo tính chuẩn xác cho các phân phối lệch, tránh nhiễu từ các nghiên cứu có kết quả cực đoan.
print("\n--- Levene\'s Test (equality of variances) ---")
print(stats.levene(low, high, center="median"))


--- Levene's Test (equality of variances) ---
LeveneResult(statistic=np.float64(0.09876327435432952), pvalue=np.float64(0.7538039677922802))


In [6]:
# [KỸ THUẬT]: Welch's T-test (equal_var=False) để xử lý sai số khi phương sai/cỡ mẫu không đồng nhất.
# [MỤC ĐÍCH]: Kiểm chứng sự khác biệt có ý nghĩa thống kê giữa Low và High Volume, chốt lại giả thuyết về ngưỡng Volume tối ưu.
print("\n--- Welch t-test (Low vs High Volume) ---")
print(stats.ttest_ind(low, high, equal_var=False))


--- Welch t-test (Low vs High Volume) ---
TtestResult(statistic=np.float64(-3.431866371515042), pvalue=np.float64(0.0008894621523165667), df=np.float64(94.92155110250833))


## 2. One-way ANOVA: Low / Moderate / High Volume
**H0:** Mean hedges_g bằng nhau ở 3 nhóm

**H1:** Ít nhất 1 nhóm khác biệt

In [7]:
# [MỤC ĐÍCH]: So sánh tác động của 3 mức Volume (Thấp - Vừa - Cao) lên kết quả phì đại.
# [ĐIỂM CHẠM HỌC THUẬT]: Kiểm chứng quy luật Dose-Response xuyên suốt các ngưỡng tập luyện để tìm ra "điểm ngọt" (optimal point) giúp tối ưu hóa cơ bắp.
g_low = df[df["volume_category"] == "Low"]["hedges_g"].dropna()
g_mod = df[df["volume_category"] == "Moderate"]["hedges_g"].dropna()
g_high= df[df["volume_category"] == "High"]["hedges_g"].dropna()

# Quan sát sơ bộ sự thay đổi của Mean (độ lớn hiệu ứng) qua từng nấc thang Volume.
print(f"Low:      n={len(g_low)},  mean={g_low.mean():.4f}")
print(f"Moderate: n={len(g_mod)},  mean={g_mod.mean():.4f}")
print(f"High:     n={len(g_high)}, mean={g_high.mean():.4f}")

# One-way ANOVA: Xác nhận sự khác biệt trung bình giữa 3 nhóm có mang ý nghĩa thống kê tổng quát hay không.
print("\n--- One-way ANOVA ---")
print(stats.f_oneway(g_low, g_mod, g_high))

Low:      n=48,  mean=0.3051
Moderate: n=61,  mean=0.3624
High:     n=89, mean=0.5416

--- One-way ANOVA ---
F_onewayResult(statistic=np.float64(8.392016271648464), pvalue=np.float64(0.0003189967833903542))


## 3. Tukey HSD Post-hoc
Xác định cặp nhóm nào khác biệt có ý nghĩa sau ANOVA

In [8]:
# [MỤC ĐÍCH]: Kiểm định hậu định (Post-hoc) bằng Tukey HSD sau khi ANOVA cho kết quả có ý nghĩa.
# [ĐIỂM CHẠM HỌC THUẬT]: Xác định chính xác "cặp" mức độ Volume nào tạo ra sự khác biệt (ví dụ: Low vs High hay Mod vs High). Điều này giúp tránh kết luận vội vàng và tìm ra ngưỡng tập luyện thực sự hiệu quả.
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# So sánh cặp (Pairwise comparison) để kiểm soát tỷ lệ lỗi loại I khi thực hiện nhiều kiểm định cùng lúc.
tukey = pairwise_tukeyhsd(
    endog=df["hedges_g"].dropna(),
    groups=df.loc[df["hedges_g"].notna(), "volume_category"],
    alpha=ALPHA
)

# Xuất bảng tóm tắt: Tập trung vào cột 'reject' (True/False) để chốt các cặp biến có sự khác biệt về mặt thống kê.
print(tukey.summary())

 Multiple Comparison of Means - Tukey HSD, FWER=0.05  
group1  group2  meandiff p-adj   lower   upper  reject
------------------------------------------------------
  High      Low  -0.2365 0.0008 -0.3873 -0.0856   True
  High Moderate  -0.1791  0.008 -0.3191 -0.0391   True
   Low Moderate   0.0573 0.6829 -0.1052  0.2198  False
------------------------------------------------------


## 4. Chi-square: failure_binary vs volume_category
**H0:** failure_binary và volume_category độc lập

In [9]:
# [MỤC ĐÍCH]: Kiểm định mối liên quan giữa Khối lượng tập và Ngưỡng thất bại (Chi-square test of independence).
# [ĐIỂM CHẠM HỌC THUẬT]: Xác định xem hai biến này có độc lập hay không, hay xu hướng tập đến "Failure" thường tập trung ở một phân khúc Volume nhất định (ví dụ: tập ít thì thường đẩy đến ngưỡng thất bại).
ct = pd.crosstab(df["volume_category"], df["failure_binary"],
                rownames=["Volume Category"],
                colnames=["Failure Binary"])
print("Contingency Table:")
print(ct)

# Kiểm định Chi-square: Dùng để phát hiện "mối liên kết ngầm" giữa các biến phân loại.
# Nếu p < Alpha, ta bác bỏ giả thuyết độc lập, khẳng định phân phối của Failure bị ảnh hưởng bởi Volume.
print("\n--- Chi-square Test ---")
chi2, p, dof, exp = stats.chi2_contingency(ct)
print(f"Chi2={chi2:.3f}, p={p:.4f}, dof={dof}")

Contingency Table:
Failure Binary    0   1
Volume Category        
Low              11  37
Moderate          9  52
High             19  70

--- Chi-square Test ---
Chi2=1.410, p=0.4940, dof=2


## 5. t-test: Trained vs Untrained
**H0:** Mean hedges_g bằng nhau giữa 2 nhóm

In [10]:
# [MỤC ĐÍCH]: So sánh hiệu quả phì đại giữa người đã tập luyện (Trained) và người mới (Untrained).
# [ĐIỂM CHẠM HỌC THUẬT]: Kiểm chứng hiện tượng "Newbie gains". Người chưa tập thường có biên độ thích nghi (mean) cao hơn do cơ thể chưa tối ưu hóa thần kinh và sinh học.
trained   = df[df["train.status"] == "trained"]["hedges_g"].dropna()
untrained = df[df["train.status"] == "untrained"]["hedges_g"].dropna()

print(f"Trained:   n={len(trained)},  mean={trained.mean():.4f}, SD={trained.std():.4f}")
print(f"Untrained: n={len(untrained)}, mean={untrained.mean():.4f}, SD={untrained.std():.4f}")

# Kiểm định tính đồng nhất phương sai. Sử dụng "median" để giảm độ nhạy với các nghiên cứu có kết quả biến thiên mạnh.
print("\n--- Levene\'s Test ---")
print(stats.levene(trained, untrained, center="median"))

# Welch t-test: Xử lý rủi ro sai số do sự khác biệt lớn về số lượng mẫu (n) giữa hai nhóm đối tượng.
print("\n--- Welch t-test (Trained vs Untrained) ---")
print(stats.ttest_ind(trained, untrained, equal_var=False))

Trained:   n=117,  mean=0.4660, SD=0.3427
Untrained: n=81, mean=0.3757, SD=0.4019

--- Levene's Test ---
LeveneResult(statistic=np.float64(0.11774333435303615), pvalue=np.float64(0.7318627883070897))

--- Welch t-test (Trained vs Untrained) ---
TtestResult(statistic=np.float64(1.648196208687475), pvalue=np.float64(0.10135201560372156), df=np.float64(153.93185310477014))


## 6. Summary — Kết luận H0/H1

In [11]:
# [MỤC ĐÍCH]: Tổng hợp toàn bộ kết quả kiểm định (T-test, ANOVA, Chi-square) vào một báo cáo duy nhất.
# [KỸ THUẬT]: Áp dụng Dictionary Mapping để quản lý kết quả. Cơ chế tự động đối chiếu p-value với ngưỡng Alpha giúp xuất quyết định bác bỏ/chấp nhận giả thuyết (Reject/Fail to reject H0) một cách khách quan, loại bỏ sai sót do quan sát thủ công.

t1 = stats.ttest_ind(low, high, equal_var=False)
f_stat, p_anova = stats.f_oneway(g_low, g_mod, g_high)
t2 = stats.ttest_ind(trained, untrained, equal_var=False)
chi2, p_chi, dof, _ = stats.chi2_contingency(
    pd.crosstab(df["volume_category"], df["failure_binary"]))

print("="*55)
print("INFERENTIAL STATISTICS SUMMARY")
print("="*55)

# Cấu trúc hóa dữ liệu kết quả kèm theo ý nghĩa sinh lý học tương ứng để báo cáo trở nên trực quan hơn.
results = {
    "t-test Low vs High":          (t1.pvalue,   "hedges_g differs by volume"),
    "ANOVA (3 groups)":            (p_anova,     "at least one group differs"),
    "Chi-sq failure vs vol":       (p_chi,       "failure & volume associated"),
    "t-test Trained vs Untrained": (t2.pvalue, "hedges_g differs by experience"),
}

for test, (p_val, meaning) in results.items():
    decision = "REJECT H0 *" if p_val < ALPHA else "fail to reject H0"
    print(f"  {test}: p={p_val:.4f} -> {decision}")

INFERENTIAL STATISTICS SUMMARY
  t-test Low vs High: p=0.0009 -> REJECT H0 *
  ANOVA (3 groups): p=0.0003 -> REJECT H0 *
  Chi-sq failure vs vol: p=0.4940 -> fail to reject H0
  t-test Trained vs Untrained: p=0.1014 -> fail to reject H0
